# 27 · Manuscript figures after review cycle 1 (Figures 1 to 7, Supplementary Figures 9 to 13)

Draws every main-text figure in one style, plus the supplementary figures for the analyses added after review. No model is run. Per-image numbers (Figures 4 to 6) are recomputed from the saved predictions exactly as in `FIGURES_final_v4`; everything else is read from saved aggregate results, and the last cell checks the recomputed numbers against `final_figures_data_v2.json`.

**What changes from the submitted figures**
- **Figure 1:** adds the 12 DDR-trained models and the BRSET test set (45 retinal models); every test set tagged "dated plan" instead of "pre-registered".
- **Figure 2:** adds BRSET; the EyePACS row is labelled as an exploratory PDR comparison (open marker), not as pre-specified, and the pre-specified EyePACS endpoint is printed under the panel.
- **Figure 3:** restores panel d (retina, BRSET), read from `brset_results_v6.json`.
- **Figure 4:** panel a relabelled: the curves are the identity for cases the union adds at shared-miss ratios 2, 5 and 10, not net benefit.
- **Figure 5:** the relative-rule line is green, as the legend says.
- **Figure 6:** "deployed system" becomes "first system".
- **Figure 7:** redrawn as the decision protocol of the revised Discussion (strength first; pilot of about 100 + 100; pick if clearly stronger, otherwise pair), with regret panels from notebooks 23 and 24.
- **Supplementary Figure 9:** scenario A and B labels, "modelled severe-visual-loss events".
- **New:** Supplementary Figures 10 (regret), 11 (pilot test, then pick or pair), 12 (deployment checks) and 13 (pooled training).

Older versions are renamed with the prefix `SUPERSEDED_` (a timestamp is added if one already exists) in `05_figures/manuscript_final/`.

BRSET is credentialed data: it is read only inside this runtime, and only figures and aggregate numbers are written. CPU, High-RAM; about 20 to 30 minutes.

In [2]:
# Cell 1: load both domains
from google.colab import drive
drive.mount('/content/drive')
import os, json, itertools, datetime, warnings
warnings.filterwarnings('ignore', category=RuntimeWarning)
import numpy as np, pandas as pd
from scipy.stats import rankdata, spearmanr
BASE = '/content/drive/MyDrive/stochastic_orthogonality/data/dr_ensemble_v2'; DERM = '/content/drive/MyDrive/stochastic_orthogonality/data/derm'
OUT = f'{BASE}/05_figures/manuscript_final'; os.makedirs(OUT, exist_ok=True)
def pick(df, c): return next(x for x in c if x in df.columns)
def tag(k): return '' if k == 0 else f'_seed{k}'
PAPER = ['densenet121_binary','densenet121_5class','efficientnet_b3_binary','efficientnet_b3_5class','resnet50_binary','resnet50_5class',
         'vit_base_binary','vit_base_5class','retfound_binary','retfound_5class','retfound_adversarial']
CK = {'densenet121': 'densenet', 'efficientnet_b3': 'effnet', 'resnet50': 'resnet', 'vit_base': 'vit', 'retfound': 'retfound'}
ck_of = lambda v: next(c for p, c in CK.items() if v.startswith(p))
def load(path, ids, col=None):
    df = pd.read_csv(path, dtype={'image_id': str}); pc = col or pick(df, ['predicted_probability','probability','prob'])
    s = df.drop_duplicates('image_id').set_index('image_id')[pc].astype(float).reindex(ids); assert s.notna().all(), path; return s.to_numpy()
DOM = {}
fr = pd.read_csv(f'{BASE}/01_splits/eyepacs_split_fresh_v2.csv', dtype={'image_id': str}).sort_values('image_id').reset_index(drop=True)
dd = pd.read_csv(f'{BASE}/01_splits/ddr_split_intraset_v2.csv', dtype={'image_id': str}).sort_values('image_id').reset_index(drop=True)
for site, df, tpl, f in (('EyePACS', fr, 'eyepacs_predictions_fresh_{m}_v2.csv', 0.25), ('DDR', dd, 'ddr_predictions_{m}_v2.csv', None)):
    ids = df.image_id.to_numpy(); g = df[pick(df, ['level','grade'])].to_numpy().astype(int)
    M = {f'{v}{tag(k)}': {'s': load(f'{BASE}/03_predictions/' + tpl.format(m=f'{v}{tag(k)}'), ids), 'ckpt': ck_of(v), 'variant': v} for v in PAPER for k in (0, 1, 2)}
    DOM[site] = {'domain': 'retina', 'models': M, 'pos': g >= 2, 'classes': {'moderate': g == 2, 'severe NPDR': g == 3, 'PDR': g == 4},
                 'f': f if f else 1.31 * float((dd.grade[dd.half == 'A'] >= 2).mean())}
CLASSES8 = ['MEL', 'BCC', 'SCC', 'AK', 'NV', 'BKL', 'DF', 'VASC']
for es in ('HIBA', 'PAD'):
    D = pd.read_csv(f'{DERM}/manifest_{es}.csv', dtype={'image_id': str, 'patient_id': str}).sort_values('image_id')
    D = D[~D.duplicated('raw_sha256', keep='first') & D.dx.isin(CLASSES8)].reset_index(drop=True); ids = D.image_id.to_numpy()
    M = {}
    for s in ('HAM10000', 'BCN20000'):
        for a in ('resnet50', 'densenet121', 'efficientnet_b3', 'convnext_t', 'vit_b', 'vit_b_dinov2'):
            for k in (0, 1): M[f'{s}_{a}_seed{k}'] = {'s': load(f'{DERM}/predictions/{es}__{s}_{a}_seed{k}.csv', ids, 'p_malignant'), 'ckpt': a, 'variant': f'{s}_{a}', 'source': s}
    mal = D.dx.isin(['MEL', 'BCC', 'SCC']).to_numpy()
    DOM[es] = {'domain': 'dermatology', 'models': M, 'pos': mal, 'classes': {'BCC': (D.dx == 'BCC').to_numpy(), 'SCC': (D.dx == 'SCC').to_numpy(), 'melanoma': (D.dx == 'MEL').to_numpy()},
               'f': 1.31 * mal.mean()}
# BRSET (credentialed data: stays in this runtime; only figures and aggregate numbers are written)
import hashlib
def sha(p, chunk=1 << 22):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(chunk), b''): h.update(b)
    return h.hexdigest()
BR = '/content/drive/MyDrive/stochastic_orthogonality/data/brset'
assert sha(f'{BR}/label_brset.csv') == json.load(open(f'{BR}/brset_access_record.json'))['label_file_sha256']
L = pd.read_csv(f'{BR}/label_brset.csv', dtype=str); gB = pd.to_numeric(L['DR_ICDR'], errors='coerce')
L = L[gB.isin([0, 1, 2, 3, 4])].copy(); L['grade'] = gB[gB.isin([0, 1, 2, 3, 4])].astype(int)
L['image_id'] = L['image_id'].astype(str).str.replace(r'\.(jpg|jpeg|png)$', '', regex=True); L = L.sort_values('image_id').reset_index(drop=True)
M = {}
for s in ('eyepacs', 'ddr'):
    for a in ('densenet121_5class', 'efficientnet_b3_5class', 'resnet50_5class', 'retfound_5class'):
        for k in (0, 1, 2):
            M[f'{s}_{a}{tag(k)}'] = {'s': load(f'{BASE}/03_predictions/brset_predictions_{s}_{a}{tag(k)}_v2.csv', L.image_id.to_numpy()), 'ckpt': a, 'variant': f'{s}_{a}', 'source': s}
gb = L.grade.to_numpy()
DOM['BRSET'] = {'domain': 'retina', 'models': M, 'pos': gb >= 2, 'classes': {'moderate': gb == 2, 'severe NPDR': gb == 3, 'PDR': gb == 4}, 'f': 1.31 * float((gb >= 2).mean())}
for k, v in DOM.items(): print(k, len(v['models']), 'models, f =', round(v['f'], 3), {c: int(m.sum()) for c, m in v['classes'].items()})

Mounted at /content/drive
EyePACS 33 models, f = 0.25 {'moderate': 2373, 'severe NPDR': 384, 'PDR': 328}
DDR 33 models, f = 0.555 {'moderate': 4477, 'severe NPDR': 236, 'PDR': 913}
HIBA 24 models, f = 0.609 {'BCC': 340, 'SCC': 158, 'melanoma': 253}
PAD 24 models, f = 0.62 {'BCC': 839, 'SCC': 190, 'melanoma': 52}
BRSET 24 models, f = 0.074 {'moderate': 451, 'severe NPDR': 78, 'PDR': 396}


In [3]:
# Cell 2: per ordered pair and class: partner's own miss rate, union rescue share, matched-volume rescue share; pair relationship and referral agreement
def refer(s, f):
    r = np.zeros(len(s), bool); r[np.argsort(-s, kind='stable')[:int(round(f * len(s)))]] = True; return r
def kappa(a, b):
    po = (a == b).mean(); pa, pb = a.mean(), b.mean(); pe = pa * pb + (1 - pa) * (1 - pb); return (po - pe) / (1 - pe)
def relation(site, A, B):
    if 'source' not in A:
        if A['ckpt'] != B['ckpt']: return 'different design'
        return 'seed sibling' if A['variant'] == B['variant'] else 'variant sibling'
    if A['variant'] == B['variant']: return 'seed sibling'
    if A['ckpt'] == B['ckpt']: return 'same checkpoint, different data'
    return 'different design, same data' if A['source'] == B['source'] else 'different design, different data'
ROWS = []
for site, S in DOM.items():
    M = S['models']; names = list(M); n = len(S['pos'])
    R = {m: refer(M[m]['s'], S['f']) for m in names}; RK = {m: rankdata(M[m]['s']) / n for m in names}
    for a, b in itertools.permutations(names, 2):
        e = refer(np.maximum(RK[a], RK[b]), S['f']); rel = relation(site, M[a], M[b]); kap = kappa(R[a], R[b])
        for c, y in S['classes'].items():
            ma, mb = y & ~R[a], y & ~R[b]; na = ma.sum()
            if na == 0: continue
            ROWS.append({'site': site, 'domain': S['domain'], 'a': a, 'b': b, 'relationship': rel, 'kappa': kap, 'class': c,
                         'partner_miss_rate': mb.sum() / y.sum(), 'anchor_miss_rate': na / y.sum(),
                         'union_rescue_share': (ma & ~mb).sum() / na, 'mv_rescue_share': (na - (y & ~e).sum()) / na,
                         'lift': ((ma & mb).sum() / y.sum()) / ((na / y.sum()) * (mb.sum() / y.sum())) if mb.sum() else np.nan})
PR = pd.DataFrame(ROWS); print(PR.groupby(['site', 'class']).size().to_string())
# rank correlation for every unordered pair (Figure 5, lower row)
PK = PR.drop_duplicates(['site', 'a', 'b'])[['site', 'domain', 'a', 'b', 'relationship', 'kappa']]; PK = PK[PK.a < PK.b].copy()
RANKS = {}
for site, S in DOM.items():
    n = len(S['pos']); RK = {m: rankdata(S['models'][m]['s']) / n for m in S['models']}
    RANKS[site] = {(a, b): float(np.corrcoef(RK[a], RK[b])[0, 1]) for a, b in itertools.combinations(sorted(S['models']), 2)}
PK['rank_corr'] = [RANKS[r.site][(r.a, r.b)] for r in PK.itertuples()]
CLASSN = {(site, c): int(m.sum()) for site, S in DOM.items() for c, m in S['classes'].items()}
print('pairs:', PK.groupby('site').size().to_dict())

site     class      
BRSET    PDR             552
         moderate        552
         severe NPDR     529
DDR      PDR            1056
         moderate       1056
         severe NPDR     960
EyePACS  PDR            1056
         moderate       1056
         severe NPDR    1056
HIBA     BCC             552
         SCC             552
         melanoma        552
PAD      BCC             552
         SCC             552
         melanoma        552
pairs: {'BRSET': 276, 'DDR': 528, 'EyePACS': 528, 'HIBA': 276, 'PAD': 276}


In [4]:
# Shared style
import shutil
import matplotlib as mpl; mpl.use('Agg'); import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
mpl.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 8.5, 'axes.spines.top': False, 'axes.spines.right': False,
                     'axes.edgecolor': '#333333', 'axes.linewidth': 0.8, 'xtick.color': '#333333', 'ytick.color': '#333333',
                     'axes.labelcolor': '#222222', 'xtick.labelsize': 8, 'ytick.labelsize': 8, 'axes.labelsize': 8.5, 'legend.frameon': False})
BLUE, SKY, ORANGE, VERM, PINK, GREY, INK, GREEN = '#0072B2', '#56B4E9', '#E69F00', '#D55E00', '#CC79A7', '#8C8C8C', '#222222', '#009E73'
DOMCOL = {'retina': BLUE, 'dermatology': VERM}
SITES = ['EyePACS', 'DDR', 'BRSET', 'HIBA', 'PAD']
SNAME = {'EyePACS': 'EyePACS (US)', 'DDR': 'DDR (China)', 'BRSET': 'BRSET (Brazil)', 'HIBA': 'HIBA (Argentina)', 'PAD': 'PAD-UFES-20 (Brazil)'}
SDOM = {'EyePACS': 'retina', 'DDR': 'retina', 'BRSET': 'retina', 'HIBA': 'dermatology', 'PAD': 'dermatology'}
MARK = {'EyePACS': 'o', 'DDR': 's', 'BRSET': 'P', 'HIBA': '^', 'PAD': 'D'}
LS = {'EyePACS': '-', 'DDR': '--', 'BRSET': ':', 'HIBA': '-', 'PAD': '--'}
CNAME = {'moderate': 'Moderate NPDR', 'severe NPDR': 'Severe NPDR', 'PDR': 'PDR', 'BCC': 'BCC', 'SCC': 'SCC', 'melanoma': 'Melanoma'}
# relationship of two systems: one display name and one colour, used in Figures 3, 5 and 6
RELMAP = {'different design, different data': 'Different design, different data', 'different design, same data': 'Different design, same data',
          'different design': 'Different design, same data', 'same checkpoint, different data': 'Same design, different data',
          'variant sibling': 'Variant sibling (same design and data)', 'seed sibling': 'Seed sibling (repeat run)'}
RELS = ['Different design, different data', 'Different design, same data', 'Same design, different data', 'Variant sibling (same design and data)', 'Seed sibling (repeat run)']
RELCOL = dict(zip(RELS, [BLUE, SKY, ORANGE, PINK, VERM]))
def ci_txt(m, lo, hi, d=2, pct=False):
    u = '%' if pct else ''
    return f'{m:.{d}f}{u} ({lo:.{d}f} to {hi:.{d}f})'
def panel(ax, letter, text, sub=None):
    ax.set_title(f'{letter}  {text}', loc='left', fontsize=9.5, weight='bold', pad=14 if sub else 6)
    if sub: ax.text(0, 1.015, sub, transform=ax.transAxes, fontsize=8, color='#444444', va='bottom')
def title(fig, t): fig.suptitle(t, x=0.01, ha='left', fontsize=11, weight='bold')
STAMP = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%d%H%M')
def save(fig, name):
    for ext in ('png', 'pdf'):
        p = f'{OUT}/{name}.{ext}'
        if os.path.exists(p):
            dest = f'{OUT}/SUPERSEDED_{name}.{ext}'
            if os.path.exists(dest): dest = f'{OUT}/SUPERSEDED_{name}_{STAMP}.{ext}'
            shutil.move(p, dest)
    fig.savefig(f'{OUT}/{name}.png', dpi=300, bbox_inches='tight'); fig.savefig(f'{OUT}/{name}.pdf', bbox_inches='tight'); plt.close(fig); print('saved', name)
PK['rel'] = PK.relationship.map(RELMAP); PR['rel'] = PR.relationship.map(RELMAP)
DR = json.load(open(f'{DERM}/derm_results_v1.json'))
RESULTS = f'{BASE}/04_results'
REV = f'{RESULTS}/revisions_c1'
REG = json.load(open(f'{REV}/regret_c1.json'))['results']; ADA = json.load(open(f'{REV}/adaptive_rule_c1.json'))['results']
DEP = json.load(open(f'{REV}/deployment_c1.json')); POOL = json.load(open(f'{REV}/pooled_vs_pairing_c1.json'))
PAIRS = pd.read_csv(f'{REV}/regret_c1_pairs.csv'); BRR = json.load(open(f'{RESULTS}/brset_results_v6.json'))
from matplotlib.patches import FancyBboxPatch
def box(ax, x, y, w, h, head, body, ec, fc, hs=10, bs=8.3, lw=1.4, ital=False):
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.004,rounding_size=0.01', ec=ec, fc=fc, lw=lw, transform=ax.transAxes))
    if head: ax.text(x + w / 2, y + h - 0.016, head, ha='center', va='top', fontsize=hs, weight='bold', color=INK, transform=ax.transAxes)
    if body: ax.text(x + w / 2, (y + h - 0.05) if head else (y + h / 2), body, ha='center', va='top' if head else 'center', fontsize=bs, color=INK,
                     linespacing=1.35, style='italic' if ital else 'normal', transform=ax.transAxes)
def arrow(ax, x0, y0, x1, y1, col='#666666'):
    ax.annotate('', xy=(x1, y1), xytext=(x0, y0), xycoords='axes fraction', arrowprops=dict(arrowstyle='-|>', color=col, lw=1.3, shrinkA=0, shrinkB=0))
def gtag(ax, x, y, t='confirmatory, dated plan'):
    ax.text(x, y, t, ha='center', va='center', fontsize=7.4, color='white', weight='bold', transform=ax.transAxes, bbox=dict(boxstyle='round,pad=0.3', fc=GREEN, ec='none'))
def blank(ax): ax.set_axis_off(); ax.set_xlim(0, 1); ax.set_ylim(0, 1)
LBL = {'EyePACS_severe': 'EyePACS, severe DR', 'DDR_severe': 'DDR, severe DR', 'DDR_pdr': 'DDR, PDR', 'BRSET_severe': 'BRSET, severe DR',
       'BRSET_mixed_severe': 'BRSET, mixed sources, severe DR', 'HIBA_mel': 'HIBA, melanoma', 'HIBA_mal': 'HIBA, any skin cancer',
       'PAD_mel': 'PAD-UFES-20, melanoma', 'PAD_mal': 'PAD-UFES-20, any skin cancer'}
LBL_DEP = {**LBL, 'DDR_pdr': 'DDR, PDR'}
KC = {'EyePACS_severe': '#0072B2', 'DDR_severe': '#56B4E9', 'DDR_pdr': '#009E73', 'BRSET_severe': '#003A70', 'BRSET_mixed_severe': '#8C8C8C',
      'HIBA_mel': '#D55E00', 'HIBA_mal': '#E69F00', 'PAD_mel': '#CC79A7', 'PAD_mal': '#8B4513'}
KM = lambda k: '^' if k.split('_')[0] in ('HIBA', 'PAD') else 'o'
def p0(x): return f'{int(np.floor(100 * x + 0.5))}%'   # percentages rounded half up, as in the text


In [5]:
# Figure 1: study design
fig = plt.figure(figsize=(17, 9.2)); ax = fig.add_axes([0, 0, 1, 0.95]); blank(ax)
RF, RE, SF, SE = '#EAF2FA', BLUE, '#FCF1E4', VERM
ax.text(0.30, 0.955, 'Diabetic retinopathy', ha='center', fontsize=15, weight='bold', color=BLUE, transform=ax.transAxes)
ax.text(0.81, 0.955, 'Skin cancer', ha='center', fontsize=15, weight='bold', color=VERM, transform=ax.transAxes)
ax.plot([0.607, 0.607], [0.21, 0.93], color='#C8C8C8', lw=1.2, transform=ax.transAxes)
# retina training and models
box(ax, 0.03, 0.78, 0.26, 0.12, 'Training: EyePACS (US screening)', '13,532 images; 2,900 for\ncheckpoint selection; split by patient', RE, RF)
box(ax, 0.33, 0.78, 0.25, 0.12, 'Training: DDR (China)', 'official training file, 6,210 images;\n2,487 for checkpoint selection', RE, RF)
box(ax, 0.03, 0.56, 0.26, 0.15, '33 models', '11 designs x 3 training runs\nDenseNet, EfficientNet, ResNet, ViT\n(2- and 5-class); 3 RETFound variants', RE, RF)
box(ax, 0.33, 0.56, 0.25, 0.15, '12 models', '4 five-class designs x 3 runs\n(DenseNet, EfficientNet, ResNet,\nRETFound); identical recipe', RE, RF)
arrow(ax, 0.16, 0.78, 0.16, 0.71); arrow(ax, 0.455, 0.78, 0.455, 0.71)
TX = [0.01, 0.155, 0.30, 0.445]; TW = 0.135
TESTS = [('EyePACS', 'test set\n15,794 eyes\n7,897 patients\nnever used\nin training'), ('DDR', '12,422 images\nChinese\nhospital\nexternal'),
         ('EyePACS', 'partner-choice\npools\n35,718 and\n17,858 images'), ('BRSET', 'Brazil\n16,266 images\n8,524 patients\nall 45 models;\nexternal to all')]
for x, (h, b) in zip(TX, TESTS):
    box(ax, x, 0.26, TW, 0.2, h, b, RE, RF, hs=10.5, bs=8.4); gtag(ax, x + TW / 2, 0.232)
for x in TX: arrow(ax, 0.16, 0.56, x + TW / 2, 0.46)
arrow(ax, 0.455, 0.56, TX[3] + TW / 2 + 0.01, 0.46)
# skin
box(ax, 0.63, 0.78, 0.17, 0.12, 'HAM10000', 'dermoscopy\nAustria, Australia\n11,720 images', SE, SF)
box(ax, 0.82, 0.78, 0.17, 0.12, 'BCN20000', 'dermoscopy\nSpain\n18,946 images', SE, SF)
box(ax, 0.665, 0.56, 0.29, 0.15, '24 models', '6 designs x 2 training collections x 2 runs\nResNet, DenseNet, EfficientNet, ConvNeXt,\nViT (supervised), ViT (DINOv2)', SE, SF)
arrow(ax, 0.715, 0.78, 0.76, 0.71); arrow(ax, 0.905, 0.78, 0.86, 0.71)
box(ax, 0.63, 0.26, 0.17, 0.2, 'HIBA', 'Argentina\n1,616 images\n623 patients\ndermoscopic and clinical', SE, SF, hs=10.5, bs=8.4)
box(ax, 0.82, 0.26, 0.17, 0.2, 'PAD-UFES-20', 'Brazil, smartphone\n2,283 images\n1,373 patients\ncancers biopsy-proven', SE, SF, hs=10.5, bs=8.4)
arrow(ax, 0.76, 0.56, 0.715, 0.46); arrow(ax, 0.86, 0.56, 0.905, 0.46)
gtag(ax, 0.715, 0.232); gtag(ax, 0.905, 0.232)
box(ax, 0.03, 0.03, 0.94, 0.15, 'Rules applied to every comparison',
    'A pair always refers the same number of cases as one system. Each analysis plan was dated before its data were examined; each test set was first scored under its confirmatory plan.\n'
    'Referral share: 25% in EyePACS; elsewhere 1.31 x the proportion of referable cases (DDR 56%, BRSET 7.5%, HIBA 61%, PAD 62%). Intervals resample patients (image groups in DDR).\n'
    'Comparators: the average member (either system, strength unknown) and the better member (chosen with hindsight).', '#9A9A9A', '#F4F4F4', hs=11, bs=9.2)
title(fig, 'Figure 1. Study design'); save(fig, 'Fig1_design')

saved Fig1_design


In [6]:
# Figure 2: benefit at the same referral volume
H1 = DR['H1_benefit']; b2 = BRR['B2_benefit']['a_eyepacs_all_110_pairings']['sev']
def pct(r, k='relative_reduction_matched_volume'): return 100 * r[k], 100 * r['ci95'][0], 100 * r['ci95'][1]
ROWS2 = [('EyePACS (US)\nPDR, vs better member', (22, 12, 34), 'retina', False),       # exploratory; Results text
         ('EyePACS (US)\nPDR, vs average member', (40, 30, 51), 'retina', False),      # secondary; Results text
         ('DDR (China)\nProliferative DR', (27, 19, 40), 'retina', True),                # pre-specified; Results text
         ('BRSET (Brazil)\nSevere DR (grade 3 or 4)', pct(b2, 'relative_reduction'), 'retina', True),
         ('HIBA (Argentina)\nAny skin cancer', pct(H1['HIBA_f']['mal']), 'dermatology', True),
         ('HIBA (Argentina)\nMelanoma', pct(H1['HIBA_f']['mel']), 'dermatology', True),
         ('PAD-UFES-20 (Brazil)\nAny skin cancer', pct(H1['PAD_f']['mal']), 'dermatology', True)]
fig, axes = plt.subplots(1, 2, figsize=(14.5, 5.2), gridspec_kw={'width_ratios': [1.15, 1]})
ax = axes[0]
for i, (lab, (m, lo, hi), dom, pre) in enumerate(ROWS2):
    c = DOMCOL[dom]; ax.plot([lo, hi], [i, i], color=c, lw=2.4, solid_capstyle='round')
    ax.plot(m, i, 'o', ms=8.5, color=c if pre else 'white', mec=c, mew=1.8, zorder=3)
    ax.text(hi + 1.2, i, f'{m:.0f}% ({lo:.0f} to {hi:.0f})', va='center', fontsize=8.5)
ax.set_yticks(range(len(ROWS2))); ax.set_yticklabels([r[0] for r in ROWS2]); ax.invert_yaxis(); ax.axvline(0, color='#333333', lw=0.8)
ax.set_xlim(-2, 62); ax.set_xlabel('Reduction in missed cases at the same referral volume (%)')
ax.legend(handles=[Line2D([], [], marker='o', ls='', color=GREY, ms=7, label='pre-specified, vs average member'),
                   Line2D([], [], marker='o', ls='', mfc='white', mec=GREY, mew=1.6, ms=7, label='exploratory or secondary')], fontsize=7.8, loc='lower right')
panel(ax, 'a', 'Against the average member, and EyePACS against the better member',
      'Pre-specified EyePACS endpoint (all severe DR, vs better member): 4.74 net misses avoided per pair (2.31 to 6.15)')
ax = axes[1]
SP = [('HIBA (Argentina)\nAny skin cancer', (17, 12, 22), (27, 18, 37)), ('HIBA (Argentina)\nMelanoma', (7, 2, 12), (26, 14, 40)),
      ('PAD-UFES-20 (Brazil)\nAny skin cancer', (10, 7, 12), (20, 14, 26))]
for i, (lab, avg, best) in enumerate(SP):
    for (m, lo, hi), dy, mk, c in ((avg, -0.17, 'o', GREY), (best, 0.17, 'D', VERM)):
        ax.plot([lo, hi], [i + dy] * 2, color=c, lw=2.4, solid_capstyle='round'); ax.plot(m, i + dy, mk, ms=8, color=c, mec='white', mew=0.8, zorder=3)
        if mk == 'D': ax.text(hi + 1.2, i + dy, f'{m:.0f}% ({lo:.0f} to {hi:.0f})', va='center', fontsize=8.5, color=VERM)
ax.set_yticks(range(len(SP))); ax.set_yticklabels([s[0] for s in SP]); ax.invert_yaxis(); ax.axvline(0, color='#333333', lw=0.8); ax.set_xlim(-2, 55)
ax.set_xlabel('Reduction in missed cases vs the average member (%)')
ax.legend(handles=[Line2D([], [], marker='o', ls='', color=GREY, ms=7, label='average over all pairs'),
                   Line2D([], [], marker='D', ls='', color=VERM, ms=7, label='strongest partner (by its own validation score)')], fontsize=7.8, loc='upper center', bbox_to_anchor=(0.5, -0.13), ncol=2)
panel(ax, 'b', 'Exploratory: choosing the strongest partner (skin)')
title(fig, 'Figure 2. A second reader reduces missed disease at the same referral volume (95% intervals resample patients or image groups)')
fig.tight_layout(); save(fig, 'Fig2_benefit')
F2 = {r[0].replace('\n', ', '): [round(v, 1) for v in r[1]] for r in ROWS2}; print(json.dumps(F2, indent=1))

saved Fig2_benefit
{
 "EyePACS (US), PDR, vs better member": [
  22,
  12,
  34
 ],
 "EyePACS (US), PDR, vs average member": [
  40,
  30,
  51
 ],
 "DDR (China), Proliferative DR": [
  27,
  19,
  40
 ],
 "BRSET (Brazil), Severe DR (grade 3 or 4)": [
  24.7,
  19.7,
  30.8
 ],
 "HIBA (Argentina), Any skin cancer": [
  16.7,
  12.2,
  21.6
 ],
 "HIBA (Argentina), Melanoma": [
  6.8,
  2.1,
  12.2
 ],
 "PAD-UFES-20 (Brazil), Any skin cancer": [
  9.9,
  7.4,
  12.4
 ]
}


In [7]:
# Figure 3: shared misses by relationship (skin a to c; retina d, BRSET), confirmatory
TYPES = ['seed_siblings', 'diff_arch_same_source', 'same_arch_diff_source', 'diff_arch_diff_source']
TLAB = {'seed_siblings': 'Same design, same data\n(repeat runs)', 'diff_arch_same_source': 'Different design,\nsame data',
        'same_arch_diff_source': 'Same design,\ndifferent data', 'diff_arch_diff_source': 'Different design,\ndifferent data'}
TCOL = {'seed_siblings': RELCOL['Seed sibling (repeat run)'], 'diff_arch_same_source': RELCOL['Different design, same data'],
        'same_arch_diff_source': RELCOL['Same design, different data'], 'diff_arch_diff_source': RELCOL['Different design, different data']}
fig, axes2 = plt.subplots(2, 2, figsize=(13, 7.8)); axes = axes2.flat
F3 = {}
RET = {'seed_siblings': 'repeat_runs', 'diff_arch_same_source': 'diff_design_same_source', 'same_arch_diff_source': 'same_design_diff_source', 'diff_arch_diff_source': 'diff_design_diff_source',
       'diff_arch_same_source__minus__diff_arch_diff_source': 'diff_design_same_source__minus__diff_design_diff_source'}
BRH3 = {k: BRR['B1_training_data']['referable'][v] for k, v in RET.items()}
for ax, key, lt, ttl in zip(axes, ('HIBA_mel', 'HIBA_mal', 'PAD_mal', 'BRSET'), 'abcd', ('HIBA (Argentina): melanoma, skin primary', 'HIBA (Argentina): any skin cancer', 'PAD-UFES-20 (Brazil): any skin cancer', 'BRSET (Brazil): referable DR, retina primary')):
    r = BRH3 if key == 'BRSET' else DR['H3_training_data'][key]; hi_all = max(r[t]['ci95'][1] for t in TYPES)
    for i, t in enumerate(TYPES):
        m, (lo, hi) = r[t]['mean'], r[t]['ci95']
        ax.plot([lo, hi], [i, i], color=TCOL[t], lw=2.2, solid_capstyle='round'); ax.plot(m, i, 'o', color=TCOL[t], ms=7.5, mec='white', mew=0.8, zorder=3)
        ax.text(hi + 0.04 * hi_all, i, ci_txt(m, lo, hi), va='center', fontsize=8)
    ax.set_yticks(range(4)); ax.set_yticklabels([TLAB[t] for t in TYPES] if ax in (axes2[0][0], axes2[1][0]) else [''] * 4); ax.invert_yaxis(); ax.set_ylim(3.5, -0.6)
    ax.axvline(1, color='#555555', lw=0.8, ls=':'); ax.set_xlim(0.8, hi_all * 1.42)
    if lt in 'cd': ax.set_xlabel('Shared-miss ratio (1 = independent errors)')
    c_ = r['diff_arch_same_source__minus__diff_arch_diff_source']; F3[key] = c_
    panel(ax, lt, ttl, f"Same data minus different data: {ci_txt(c_['mean'], *c_['ci95'])}")
axes2[1][1].text(0, -0.3, 'Retina: the 24 five-class systems of four designs, trained on EyePACS or on DDR', transform=axes2[1][1].transAxes, fontsize=7.8, color='#444444')
title(fig, 'Figure 3. Shared blind spots by how two systems are related (95% intervals resample patients)')
fig.tight_layout(); fig.subplots_adjust(hspace=0.55); save(fig, 'Fig3_shared_misses_by_relationship')

saved Fig3_shared_misses_by_relationship


In [8]:
# Figure 4: the miss-rate law (condition-population points with fewer than 10 misses per model omitted, as in FIGURES_brset)
PR['anchor_misses'] = PR.anchor_miss_rate * PR.apply(lambda r: CLASSN[(r.site, r['class'])], axis=1)
ND0 = PR[PR.relationship.str.startswith('different design')]
MED = ND0.groupby(['site', 'class']).anchor_misses.median()
KEEP = MED[MED >= 10].index; DROPPED = [f'{s} {c} ({m:.1f})' for (s, c), m in MED[MED < 10].items()]
ND = ND0[ND0.set_index(['site', 'class']).index.isin(KEEP)]
print('omitted (median misses per model < 10):', DROPPED)
SUMM = ND.groupby(['site', 'domain', 'class']).agg(partner_miss_rate=('partner_miss_rate', 'median'), union_rescue=('union_rescue_share', 'median'),
                                                  mv_rescue=('mv_rescue_share', 'median'), lift=('lift', 'median'), n_pairs=('a', 'size')).reset_index()
rho = spearmanr(SUMM.partner_miss_rate, SUMM.mv_rescue)[0]
fig, axes = plt.subplots(1, 2, figsize=(12, 4.6), gridspec_kw={'width_ratios': [1, 1.05]})
ax = axes[0]; smp = ND.sample(min(len(ND), 6000), random_state=1)
for dom, col in DOMCOL.items():
    s = smp[smp.domain == dom]; ax.scatter(s.partner_miss_rate, s.union_rescue_share, s=4, alpha=0.16, color=col, lw=0, rasterized=True)
p = np.linspace(0.005, 0.6, 200)
for L, ls in ((2, ':'), (5, '--'), (10, '-.')):
    y = np.clip(1 - L * p, 0, 1); ax.plot(p, y, ls, color='#444444', lw=1)
    k = np.argmin(np.abs(y - 0.08)); ax.text(p[k] + 0.008, 0.1, f'ratio {L}', fontsize=7.5, color='#444444')
ax.set_xlim(0, 0.62); ax.set_ylim(-0.02, 1.02)
ax.set_xlabel("Partner's own miss rate for that condition"); ax.set_ylabel("Share of the first system's misses the partner also flags")
ax.legend(handles=[Line2D([], [], marker='o', ls='', color=c, ms=5, label=l) for l, c in (('Retina pairs', BLUE), ('Skin pairs', VERM))], fontsize=7.5, loc='upper right')
panel(ax, 'a', 'Every pair and condition (different designs)', 'Curves: the identity at shared-miss ratios of 2, 5 and 10 (cases the union adds, not net benefit)')
LABOFF = {('DDR', 'moderate'): (-7, -9, 'right')}   # moves one label clear of its neighbour
ax = axes[1]
for _, r in SUMM.iterrows():
    x, y = r.partner_miss_rate, 100 * r.mv_rescue
    ax.scatter(x, y, s=90 if r.site == 'BRSET' else 70, marker=MARK[r.site], color=DOMCOL[r.domain], edgecolor='white', lw=0.4 if r.site == 'BRSET' else 0.8, zorder=3)
    dx, dy, ha = LABOFF.get((r.site, r['class']), (6, 2, 'left'))
    ax.annotate(CNAME[r['class']], (x, y), textcoords='offset points', xytext=(dx, dy), ha=ha, fontsize=7.2, color='#333333')
ax.set_xlabel('Median miss rate of each system for that condition'); ax.set_ylabel('Median % of misses rescued at matched volume')
ax.set_xlim(0, max(SUMM.partner_miss_rate) * 1.18); ax.set_ylim(0, max(100 * SUMM.mv_rescue) * 1.12)
h = [Line2D([], [], marker=MARK[s], ls='', color=DOMCOL[SDOM[s]], mec='white', ms=7, label=SNAME[s]) for s in SITES]
ax.legend(handles=h, fontsize=7.5, loc='upper right')
panel(ax, 'b', 'Rescue follows how rarely each system misses, not severity', f'One point per condition and population; Spearman rho {rho:.2f}')
title(fig, f'Figure 4. The benefit is largest where each reader rarely misses (retina and skin; {len(SUMM)} condition-population points)')
fig.tight_layout(); save(fig, 'Fig4_miss_rate_law')
print(SUMM.round(3).to_string())

omitted (median misses per model < 10): ['BRSET severe NPDR (5.0)', 'DDR severe NPDR (2.0)']
saved Fig4_miss_rate_law
       site       domain        class  partner_miss_rate  union_rescue  mv_rescue   lift  n_pairs
0     BRSET       retina          PDR              0.266         0.451      0.097  1.788      432
1     BRSET       retina     moderate              0.427         0.286      0.057  1.398      432
2       DDR       retina          PDR              0.053         0.560      0.250  8.669      864
3       DDR       retina     moderate              0.173         0.435      0.118  3.324      864
4   EyePACS       retina          PDR              0.055         0.650      0.400  7.130      864
5   EyePACS       retina     moderate              0.351         0.316      0.076  1.963      864
6   EyePACS       retina  severe NPDR              0.068         0.456      0.167  8.234      864
7      HIBA  dermatology          BCC              0.062         0.810      0.308  3.333      480


In [9]:
# Figure 5: related systems give themselves away (agreement by relationship, five populations; relative rule as in FIGURES_brset)
from sklearn.metrics import roc_auc_score
DET = {}
fig, axes = plt.subplots(2, 5, figsize=(17, 7.4), sharey=True)
rng = np.random.default_rng(0)
for row, (metric, lab) in enumerate((('kappa', 'Agreement on referral decisions (kappa)'), ('rank_corr', 'Agreement on score rankings (rank correlation)'))):
    for ax, site in zip(axes[row], SITES):
        s = PK[PK.site == site]; sib = s.relationship.str.contains('sibling'); dif = s.relationship.str.startswith('different design')
        thr = float(np.percentile(s[metric][dif], 95)); flag = float((s[metric][sib] > thr).mean())
        auc = float(roc_auc_score(np.r_[np.ones(sib.sum()), np.zeros(dif.sum())], np.r_[s[metric][sib], s[metric][dif]]))
        DET.setdefault(site, {})[metric] = {'auc_siblings_vs_different_design': round(auc, 3), 'relative_threshold': round(thr, 4), 'siblings_flagged': round(flag, 3)}
        for i, rl in enumerate(RELS):
            v = s[metric][s.rel == rl].to_numpy()
            if len(v) == 0:
                ax.text(0.02, i, 'not in this design', transform=ax.get_yaxis_transform(), va='center', fontsize=7, color='#9a9a9a'); continue
            bp = ax.boxplot([v], positions=[i], vert=False, widths=0.55, patch_artist=True, showfliers=False, whis=(2.5, 97.5),
                            medianprops=dict(color=INK, lw=1.4), whiskerprops=dict(color='#555555', lw=0.8), capprops=dict(color='#555555', lw=0.8),
                            boxprops=dict(facecolor=RELCOL[rl], alpha=0.35, edgecolor=RELCOL[rl], lw=1))
            jit = rng.uniform(-0.18, 0.18, len(v)) if len(v) > 1 else np.zeros(1)
            ax.scatter(v, i + jit, s=4 if len(v) > 200 else 7, color=RELCOL[rl], alpha=0.45 if len(v) > 200 else 0.8, lw=0, zorder=3, rasterized=True)
        ax.axvline(thr, color=GREEN, ls='-', lw=1.4)
        ax.set_yticks(range(len(RELS))); ax.set_yticklabels(RELS); ax.set_ylim(len(RELS) - 0.5, -0.6)
        ax.set_xlabel(lab, fontsize=8); ax.tick_params(axis='y', length=0)
        ax.set_title(f'{SNAME[site]}', loc='left', fontsize=9, weight='bold', pad=14)
        ax.text(0, 1.015, f'AUC {auc:.2f}; {100 * flag:.0f}% of siblings flagged', transform=ax.transAxes, fontsize=7.5, color='#444444', va='bottom')
for row, (lt, t) in enumerate((('a', 'Agreement on referral decisions'), ('b', 'Agreement on score rankings'))):
    axes[row][0].annotate(f'{lt}  {t}', xy=(0, 1), xycoords='axes fraction', xytext=(-205, 30), textcoords='offset points', fontsize=10, weight='bold', ha='left')
fig.legend(handles=[Line2D([], [], color=GREEN, ls='-', lw=1.4, label='Relative rule: 95th percentile of agreement among systems known to differ in design')],
           loc='lower left', bbox_to_anchor=(0.01, -0.01), fontsize=8)
title(fig, 'Figure 5. Related systems give themselves away on local unlabelled images')
fig.tight_layout(rect=(0, 0.03, 1, 0.97)); fig.subplots_adjust(hspace=0.55); save(fig, 'Fig5_siblings'); print(json.dumps(DET, indent=1))

saved Fig5_siblings
{
 "EyePACS": {
  "kappa": {
   "auc_siblings_vs_different_design": 1.0,
   "relative_threshold": 0.5979,
   "siblings_flagged": 1.0
  },
  "rank_corr": {
   "auc_siblings_vs_different_design": 1.0,
   "relative_threshold": 0.6886,
   "siblings_flagged": 1.0
  }
 },
 "DDR": {
  "kappa": {
   "auc_siblings_vs_different_design": 1.0,
   "relative_threshold": 0.6478,
   "siblings_flagged": 1.0
  },
  "rank_corr": {
   "auc_siblings_vs_different_design": 1.0,
   "relative_threshold": 0.8423,
   "siblings_flagged": 1.0
  }
 },
 "BRSET": {
  "kappa": {
   "auc_siblings_vs_different_design": 0.879,
   "relative_threshold": 0.6721,
   "siblings_flagged": 0.625
  },
  "rank_corr": {
   "auc_siblings_vs_different_design": 0.998,
   "relative_threshold": 0.659,
   "siblings_flagged": 1.0
  }
 },
 "HIBA": {
  "kappa": {
   "auc_siblings_vs_different_design": 0.947,
   "relative_threshold": 0.6856,
   "siblings_flagged": 0.833
  },
  "rank_corr": {
   "auc_siblings_vs_different_

In [10]:
# Figure 6: agreement predicts benefit (matched-volume rescue by kappa band; rescue by relationship)
BANDS = [0.0, 0.4, 0.5, 0.6, 0.7, 0.8, 1.0]
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.6), gridspec_kw={'width_ratios': [1, 1.15]}); BEN = {}
for site in SITES:
    S = DOM[site]; s = PR[(PR.site == site) & (PR['class'].isin(list(S['classes'])))]
    allc = s.groupby(pd.cut(s.kappa, BANDS), observed=False).mv_rescue_share.agg(['median', 'count']).reset_index()
    allc = allc[allc['count'] >= 10]; mid = [iv.mid for iv in allc.kappa]
    axes[0].plot(mid, 100 * allc['median'], ls=LS[site], marker=MARK[site], color=DOMCOL[SDOM[site]], lw=1.8, ms=6, mec='white', label=SNAME[site])
    BEN[site] = {str(k): {'median_rescue': round(float(m), 3), 'pairs': int(c)} for k, m, c in zip(allc.kappa, allc['median'], allc['count'])}
ax = axes[0]; ax.set_xlabel('Referral agreement with the first system (kappa, banded)'); ax.set_ylabel('Median % of misses rescued at matched volume')
ax.set_ylim(bottom=0); ax.legend(fontsize=7.5, loc='upper right')
panel(ax, 'a', 'The more a candidate agrees, the less it adds', 'All target conditions; bands with at least 10 pairs')
rels = PR.groupby(['site', 'relationship']).mv_rescue_share.median().unstack(0)
ax = axes[1]; off = dict(zip(SITES, np.linspace(-0.28, 0.28, len(SITES))))
for i, rl in enumerate(RELS):
    ax.axhspan(i - 0.42, i + 0.42, color=RELCOL[rl], alpha=0.08, lw=0)
    for site in SITES:
        raw = [k for k, v in RELMAP.items() if v == rl and k in rels.index and not np.isnan(rels.loc[k, site])]
        for k in raw: ax.scatter(100 * rels.loc[k, site], i + off[site], marker=MARK[site], color=DOMCOL[SDOM[site]], s=46, edgecolor='white', lw=0.7, zorder=3)
ax.set_yticks(range(len(RELS))); ax.set_yticklabels(RELS); ax.set_ylim(len(RELS) - 0.5, -0.5); ax.tick_params(axis='y', length=0)
ax.set_xlim(left=-0.6); ax.set_xlabel('Median % of misses rescued at matched volume')
ax.legend(handles=[Line2D([], [], marker=MARK[s], ls='', color=DOMCOL[SDOM[s]], mec='white', ms=7, label=SNAME[s]) for s in SITES], fontsize=7.5, loc='lower right')
panel(ax, 'b', 'Siblings add least; different data and design add most', 'One point per population; retinal systems were all trained on EyePACS except in BRSET')
title(fig, 'Figure 6. Agreement on local unlabelled images predicts how much a candidate can add')
fig.tight_layout(); save(fig, 'Fig6_agreement_benefit')

saved Fig6_agreement_benefit


In [11]:
# Figure 7: decision protocol, with regret panels (notebooks 23 and 24; exploratory, post hoc)
K7 = '100'
STR = [('Pick one blind', GREY, 'o'), ('Always pick the pilot winner', ORANGE, 's'), ('Always pair', BLUE, 'D'), ('Pilot test, then pick or pair', GREEN, '^')]
def strat(key):
    fs = REG[key]['full_sample']; m = fs.get('main roster', fs['all']); a = ADA[key][K7]
    return [m['blind_expected_regret'][0], a['always_pick'][0], a['always_pair'][0], a['adaptive_delong_alpha_0.05'][0]]
KEYS100 = [k for k in ADA if K7 in ADA[k]]
S100 = np.array([strat(k)[1:] for k in KEYS100]); MEAN7, WORST7 = S100.mean(0), S100.max(0)
sib_bm = BRR['B4_sibling_test']['repeat_runs_flagged']; rank_fl = [DET[s]['rank_corr']['siblings_flagged'] for s in SITES]
fig = plt.figure(figsize=(17.5, 10.2))
ax = fig.add_axes([0.0, 0.03, 0.57, 0.9]); blank(ax)
ax.text(0.81, 0.985, 'Evidence in this study', ha='center', fontsize=10.5, weight='bold', color='#666666', transform=ax.transAxes)
STEPS = [('1  Judge a second system at the referral volume you can absorb',
          'A pair refers the same number of cases as one system. Referring\nevery case either system flags catches more, but only by referring more.',
          'Same volume: fewer misses than the average member\nin all five populations. Either-flags rule (EyePACS):\n59 extra false referrals per severe eye rescued.', BLUE, '#EAF2FA'),
         ('2  Choose partners for strength before diversity',
          'Different training data means fewer shared misses, but a weaker\npartner can make a strong system worse.',
          'Fewer shared misses with different data (pre-specified,\nboth domains); no extra catch once partner strength is\nallowed for. In BRSET, a weaker DDR-trained partner left\nEyePACS-trained systems 10.6 more misses per 10,000.', BLUE, '#EAF2FA'),
         ('3  Test the candidates on local cases, then pick or pair',
          'Draw about 100 severe cases and 100 others from many patients.\nIf one system is clearly stronger (paired DeLong test, p < 0.05),\nuse it alone; otherwise run both as a pair.',
          f'Mean regret per 10,000 at 100 + 100 (worst):\npilot test {MEAN7[2]:.1f} ({WORST7[2]:.1f}); always pair {MEAN7[1]:.1f} ({WORST7[1]:.1f});\nalways pick the winner {MEAN7[0]:.1f} ({WORST7[0]:.1f}). Exploratory.', GREEN, '#E7F4EF'),
         ('4  Check for close relatives on unlabelled local images',
          "Compare how the candidates rank the programme's own images;\nflag agreement above that among systems known to differ.",
          f'Pre-specified kappa rule flagged {p0(sib_bm)} of BRSET siblings\n(target 80%); score ranks flagged {p0(min(rank_fl))} to {p0(max(rank_fl))}\nacross populations (exploratory, revised after the test).', BLUE, '#EAF2FA'),
         ('5  Set the referral share with local prevalence in mind',
          'Keep referrals in proportion to local referable disease.',
          'Neither pre-specified test found the predicted reversal\n(DDR at a copied 25% share; EyePACS enriched to\n40.6% referable).', BLUE, '#EAF2FA')]
y0, hh, gap = 0.80, 0.135, 0.045
for i, (h, b, ev, ec, fc) in enumerate(STEPS):
    y = y0 - i * (hh + gap)
    box(ax, 0.0, y, 0.6, hh, h, b, ec, fc, hs=10, bs=8.6, lw=1.6 if ec == GREEN else 1.4)
    box(ax, 0.63, y + 0.01, 0.365, hh - 0.02, None, ev, '#BDBDBD', '#F7F7F7', bs=7.9, lw=1.0, ital=True)
    ax.plot([0.6, 0.63], [y + hh / 2] * 2, color='#BDBDBD', lw=1, transform=ax.transAxes)
    if i < len(STEPS) - 1: arrow(ax, 0.3, y, 0.3, y - gap + 0.002, col=BLUE)
ax.text(0.0, y - 0.035, 'Expect the most benefit for conditions each system rarely misses; where all systems miss the same cases\n(melanoma among moles), plan another kind of second look.',
        fontsize=8.4, color='#444444', va='top', transform=ax.transAxes)
ax.text(0.0, 1.0, 'a  A protocol for choosing and checking a second AI reader', fontsize=10.5, weight='bold', va='bottom', transform=ax.transAxes)
gs = fig.add_gridspec(3, 2, left=0.63, right=0.99, top=0.9, bottom=0.11, hspace=0.75, wspace=0.32)
PTC = {'main roster': (BLUE, 'o', 'pair'), 'both EyePACS-trained': (BLUE, 'o', 'both EyePACS-trained'), 'one of each': (GREEN, '^', 'one of each'), 'both DDR-trained': (VERM, 's', 'both DDR-trained')}
for j, (key, lt, ttl) in enumerate((('EyePACS_severe', 'b', 'Close candidates: EyePACS'), ('BRSET_mixed_severe', 'c', 'Far apart: BRSET, mixed sources'))):
    a_ = fig.add_subplot(gs[0, j]); d = PAIRS[PAIRS.roster == key]; mx = d.gap_per_10k.max() * 1.05
    a_.fill_between([0, mx], [0, mx / 2], [0, mx], color='#DDDDDD', alpha=0.5, lw=0)
    a_.plot([0, mx], [0, mx / 2], ls='--', color='#777777', lw=1); a_.axhline(0, color='#333333', lw=0.8)
    for t, g_ in d.groupby('pair_type'):
        c, mk, lab = PTC[t]; a_.scatter(g_.gap_per_10k, g_.pair_regret_per_10k, s=9, alpha=0.5, color=c, marker=mk, lw=0, label=lab, rasterized=True)
    a_.set_xlabel('Gap between the two systems\n(missed per 10,000)', fontsize=7.8); a_.set_ylabel('Pair regret (missed per 10,000)' if j == 0 else '', fontsize=7.8)
    a_.set_title(f'{lt}  {ttl}', loc='left', fontsize=9, weight='bold')
    if j == 0: a_.legend(handles=[Line2D([], [], ls='--', color='#777777', label='blind pick, expected'), Patch(color='#DDDDDD', label='up to the worst-case blind pick'),
                                  Line2D([], [], ls='', marker='o', color=BLUE, ms=4, label='pair (below 0: beats the better system)')], fontsize=6.4, loc='upper left', handletextpad=0.3)
    if j == 1: a_.legend(fontsize=6.6, loc='upper left', markerscale=1.6, handletextpad=0.2)
for row, (lt, keys, ttl) in enumerate((('d', ['EyePACS_severe', 'DDR_severe', 'DDR_pdr', 'BRSET_severe', 'BRSET_mixed_severe'], 'Retina: regret by strategy, pilot 100 + 100'),
                                      ('e', ['HIBA_mel', 'HIBA_mal', 'PAD_mal'], 'Skin: regret by strategy, pilot 100 + 100')), start=1):
    a_ = fig.add_subplot(gs[row, :]); a_.axvline(0, color='#333333', lw=0.8)
    for i, k in enumerate(keys):
        v = strat(k)
        for (lab, c, mk), x, dy in zip(STR, v, (-0.27, -0.09, 0.09, 0.27)): a_.scatter(x, i + dy, color=c, marker=mk, s=34, edgecolor='white', lw=0.5, zorder=3, label=lab if i == 0 else None)
    a_.set_yticks(range(len(keys))); a_.set_yticklabels([LBL[k] for k in keys], fontsize=7.8); a_.invert_yaxis(); a_.tick_params(axis='y', length=0)
    a_.set_xlabel('Regret: missed per 10,000 above the better system (0 = knowing which is stronger)', fontsize=7.8)
    a_.set_title(f'{lt}  {ttl}', loc='left', fontsize=9, weight='bold')
    if row == 2: a_.legend(fontsize=7.2, loc='upper center', bbox_to_anchor=(0.45, -0.32), ncol=4, handletextpad=0.2, columnspacing=1.0)
title(fig, 'Figure 7. Choosing and checking a second AI reader')
save(fig, 'Fig7_decision_flow')
F7 = {'strategy_mean_k100': dict(zip([s[0] for s in STR[1:]], MEAN7.round(2).tolist())), 'strategy_worst_k100': dict(zip([s[0] for s in STR[1:]], WORST7.round(2).tolist())),
      'combinations_k100': KEYS100, 'kappa_rule_brset_siblings_flagged': sib_bm, 'rank_rule_siblings_flagged': dict(zip(SITES, rank_fl))}
print(json.dumps(F7, indent=1))

saved Fig7_decision_flow
{
 "strategy_mean_k100": {
  "Always pick the pilot winner": 7.61,
  "Always pair": -5.57,
  "Pilot test, then pick or pair": -6.07
 },
 "strategy_worst_k100": {
  "Always pick the pilot winner": 20.78,
  "Always pair": 5.75,
  "Pilot test, then pick or pair": 3.4
 },
 "combinations_k100": [
  "EyePACS_severe",
  "DDR_severe",
  "BRSET_severe",
  "DDR_pdr",
  "BRSET_mixed_severe",
  "HIBA_mel",
  "HIBA_mal",
  "PAD_mal"
 ],
 "kappa_rule_brset_siblings_flagged": 0.625,
 "rank_rule_siblings_flagged": {
  "EyePACS": 1.0,
  "DDR": 1.0,
  "BRSET": 1.0,
  "HIBA": 0.917,
  "PAD": 0.833
 }
}


In [12]:
# Supplementary Figure 9: priced false referrals, the two bounds (from notebook 18b's saved curves; aggregate numbers only)
CUR = pd.read_csv(f'{RESULTS}/bounds_B_tuned_K_v2.csv'); BEK = pd.read_csv(f'{RESULTS}/bounds_B_breakeven_K_v2.csv')
BOUNDS = [('Scenario A: better member (chosen with hindsight); each missed eye carries a modelled risk of severe visual loss', 'better', 'literature harm (central)', BLUE, '-'),
          ('Scenario B: average member; every missed severe eye counted as lost', 'deployed', 'every missed severe eye blinds', VERM, '--')]
POPS = [('eyepacs', 'EyePACS (US)'), ('ddr', 'DDR (China)'), ('brset', 'BRSET (Brazil)')]
fig, axes = plt.subplots(2, 3, figsize=(13, 6.6), sharex=True)
for row, (blab, cmp_, harm, col, ls) in enumerate(BOUNDS):
    for j, (pop, pname) in enumerate(POPS):
        ax = axes[row][j]; c = CUR[(CUR.population == pop) & (CUR.comparator == cmp_) & (CUR.harm == harm)].sort_values('K')
        ax.axvspan(800, 3200, color='#BDBDBD', alpha=0.35, lw=0)
        ax.fill_between(c.K, c.ci_lo, c.ci_hi, color=col, alpha=0.18, lw=0); ax.plot(c.K, c.advantage, color=col, ls=ls, lw=2)
        ax.axhline(0, color='#555555', lw=0.8); ax.set_xscale('log')
        b = BEK[(BEK.population == pop) & (BEK.comparator == cmp_) & (BEK.harm == harm)].iloc[0]
        ax.set_title(pname, loc='left', fontsize=9, weight='bold', pad=14)
        lab_ = 'Pair ahead at every K tested, across the 95% interval' if b.ci == 'below 10 to below 10' else f'Break-even K: {b.breakeven_K} (95% CI {b.ci})'
        ax.text(0, 1.015, lab_, transform=ax.transAxes, fontsize=7.8, color='#444444', va='bottom')
        if row == 1: ax.set_xlabel('K: false referrals accepted per modelled severe-visual-loss event')
        if j == 0: ax.set_ylabel('Pair minus comparator\n(modelled events per 10,000 eyes)')
    axes[row][0].annotate(f"{'ab'[row]}  {blab}", xy=(0, 1), xycoords='axes fraction', xytext=(-55, 34), textcoords='offset points', fontsize=9.5, weight='bold', ha='left').set_in_layout(False)
fig.legend(handles=[Patch(color='#BDBDBD', alpha=0.6, label='One specialist visit under the only-eye QALY valuation (K about 800 to 3,200)'),
                    Patch(color=GREY, alpha=0.3, label='95% interval (1,000 resamples of patients or image groups)')], loc='lower left', bbox_to_anchor=(0.01, -0.02), fontsize=8, ncol=2)
title(fig, 'Supplementary Figure 9. Priced false referrals, two harm scenarios: each strategy tunes its own referral volume (above 0, the pair is ahead)')
fig.tight_layout(rect=(0, 0.04, 1, 0.96)); fig.subplots_adjust(hspace=0.6); save(fig, 'SuppFig9_priced_false_referrals')

saved SuppFig9_priced_false_referrals


In [13]:
# Supplementary Figure 10: regret of picking one system versus pairing (notebook 23)
PANELS = ['EyePACS_severe', 'DDR_pdr', 'BRSET_severe', 'BRSET_mixed_severe', 'HIBA_mal', 'PAD_mal']
PT = {**PTC, 'same collection': (VERM, 's', 'same collection'), 'different collection': (BLUE, 'o', 'different collection')}
fig, axes = plt.subplots(2, 3, figsize=(15, 8.6))
for a_, key, lt in zip(axes.flat, PANELS, 'abcdef'):
    d = PAIRS[PAIRS.roster == key]; mx = d.gap_per_10k.max() * 1.05; fs = REG[key]['full_sample']; m = fs.get('main roster', fs['all'])
    a_.fill_between([0, mx], [0, mx / 2], [0, mx], color='#DDDDDD', alpha=0.5, lw=0)
    a_.plot([0, mx], [0, mx / 2], ls='--', color='#777777', lw=1, label='blind pick, expected (half the gap)'); a_.plot([0, mx], [0, mx], ls=':', color='#777777', lw=1, label='blind pick, worst case (whole gap)')
    a_.axhline(0, color='#333333', lw=0.8)
    for t, g_ in d.groupby('pair_type'):
        c, mk, lab = PT[t]; a_.scatter(g_.gap_per_10k, g_.pair_regret_per_10k, s=12, alpha=0.5, color=c, marker=mk, lw=0, label=lab, rasterized=True)
    a_.set_xlabel('Gap between the two systems (missed per 10,000)'); a_.set_ylabel('Pair regret (missed per 10,000)')
    panel(a_, lt, LBL[key], f"Pair {ci_txt(*m['pair_regret'])}; blind pick {ci_txt(*m['blind_expected_regret'])}")
    a_.text(0.98, 0.03, f"pair beat a blind pick in {p0(m['share_pairs_pair_beats_blind_pick'])} of pairs", transform=a_.transAxes, ha='right', fontsize=7.4, color='#444444')
    a_.legend(fontsize=6.6, loc='upper left', handletextpad=0.2)
title(fig, 'Supplementary Figure 10. Regret of picking one system blind versus pairing (each dot is one pair; regret is missed per 10,000 above the better member)')
fig.tight_layout(); save(fig, 'SuppFig10_regret')

saved SuppFig10_regret


In [14]:
# Supplementary Figure 11: pilot test, then pick or pair (notebook 24)
KS = ['25', '50', '100']; SN = [('always_pick', 'Always pick the pilot winner', ORANGE, 's'), ('always_pair', 'Always pair', BLUE, 'D'),
                               ('adaptive_delong_alpha_0.05', 'Pilot test, then pick or pair', GREEN, '^'), ('hindsight_best_of_pick_or_pair', 'Hindsight bound', GREY, 'o')]
fig, axes = plt.subplots(1, 3, figsize=(15, 4.8), gridspec_kw={'width_ratios': [1, 1, 1.25]})
S11 = {}
for k in KS:
    keys = [kk for kk in ADA if k in ADA[kk]]; S11[k] = {s: [ADA[kk][k][s][0] for kk in keys] for s, *_ in SN}
for a_, fn, lt, ttl in ((axes[0], np.mean, 'a', 'Mean regret over populations and outcomes'), (axes[1], np.max, 'b', 'Worst population and outcome')):
    for s, lab, c, mk in SN:
        if fn is np.max and s.startswith('hindsight'): continue
        a_.plot([int(k) for k in KS], [fn(S11[k][s]) for k in KS], marker=mk, color=c, lw=1.8, ms=7, mec='white', label=lab)
    a_.axhline(0, color='#333333', lw=0.8); a_.set_xticks([25, 50, 100]); a_.set_xticklabels(['25 + 25', '50 + 50', '100 + 100'])
    a_.set_xlabel('Pilot size (severe cases + others)'); a_.set_ylabel('Regret (missed per 10,000)'); panel(a_, lt, ttl)
axes[0].legend(fontsize=7.4, loc='upper right')
a_ = axes[2]
for i, kk in enumerate(ADA):
    sh = [ADA[kk][k]['share_paired_alpha_0.05'] * 100 if k in ADA[kk] else np.nan for k in KS]
    a_.plot([25, 50, 100], sh, marker=KM(kk), ms=5, lw=1.4, color=KC[kk], label=LBL[kk])
a_.set_xticks([25, 50, 100]); a_.set_xticklabels(['25 + 25', '50 + 50', '100 + 100']); a_.set_ylim(0, 105); a_.legend(fontsize=6.6, loc='lower left', ncol=2)
a_.set_xlabel('Pilot size (severe cases + others)'); a_.set_ylabel('Pilots that ended in pairing (%)'); panel(a_, 'c', 'How often the test found no clear winner')
title(fig, 'Supplementary Figure 11. A local pilot, then pick or pair (paired DeLong test, p < 0.05; scored on patients outside the pilot; 100 draws)')
fig.tight_layout(); save(fig, 'SuppFig11_pilot_then_pick_or_pair')
print(json.dumps({k: {s: [round(float(np.mean(v)), 2), round(float(np.max(v)), 2)] for s, v in S11[k].items()} for k in KS}, indent=1))

saved SuppFig11_pilot_then_pick_or_pair
{
 "25": {
  "always_pick": [
   12.21,
   39.22
  ],
  "always_pair": [
   -4.43,
   7.57
  ],
  "adaptive_delong_alpha_0.05": [
   -4.31,
   5.06
  ],
  "hindsight_best_of_pick_or_pair": [
   -9.5,
   -0.13
  ]
 },
 "50": {
  "always_pick": [
   10.58,
   29.46
  ],
  "always_pair": [
   -5.37,
   7.0
  ],
  "adaptive_delong_alpha_0.05": [
   -5.45,
   5.28
  ],
  "hindsight_best_of_pick_or_pair": [
   -10.9,
   -0.42
  ]
 },
 "100": {
  "always_pick": [
   7.61,
   20.78
  ],
  "always_pair": [
   -5.57,
   5.75
  ],
  "adaptive_delong_alpha_0.05": [
   -6.07,
   3.4
  ],
  "hindsight_best_of_pick_or_pair": [
   -11.11,
   -0.58
  ]
 }
}


In [15]:
# Supplementary Figure 12: deployment checks (notebook 25)
A_, B_, C_ = DEP['A_categorical_outputs'], DEP['B_fixed_partner'], DEP['C_ensemble_size']
fig = plt.figure(figsize=(16, 9.4)); gs = fig.add_gridspec(2, 3, height_ratios=[1, 1], hspace=0.6, wspace=0.42)
a_ = fig.add_subplot(gs[0, :2]); keys = list(A_); xs = np.arange(len(keys)); w = 0.2
for j, (lab, c, f_) in enumerate((('Maximum rank, 2 categories', '#BDBDBD', lambda o: o['L2']['share_of_benefit_kept_maxrank']),
                                 ('Maximum rank, 3 categories', SKY, lambda o: o['L3']['share_of_benefit_kept_maxrank']),
                                 ('Maximum rank, 5 categories', BLUE, lambda o: o['L5']['share_of_benefit_kept_maxrank']),
                                 ('Joint table, 2 categories, pilot 100 + 100', ORANGE, lambda o: o['L2']['pilot_100']['share_kept_table'][0]))):
    v = np.array([100 * f_(A_[k]) for k in keys]); vc = np.clip(v, -40, 140)
    a_.bar(xs + (j - 1.5) * w, vc, w, color=c, label=lab)
    for x, vv, cc in zip(xs + (j - 1.5) * w, v, vc):
        if abs(vv) < 0.5: a_.text(x, 2, '0%', ha='center', va='bottom', fontsize=6.6, color='#444444')
        if vv != cc: a_.text(x, cc + (3 if cc > 0 else -3), f'{vv:.0f}%', ha='center', va='bottom' if cc > 0 else 'top', fontsize=6.6, color='#444444')
a_.axhline(100, color='#333333', lw=0.8, ls=':'); a_.axhline(0, color='#333333', lw=0.8); a_.set_ylim(-48, 150)
a_.set_xticks(xs); a_.set_xticklabels([LBL_DEP[k].replace(', ', ',\n') for k in keys], fontsize=7.8); a_.set_ylabel('Share of the continuous-score benefit kept (%)')
a_.legend(fontsize=7.2, ncol=2, loc='upper left'); panel(a_, 'a', 'Vendors that report only categories', 'Benefit over the average member; dotted line: as much as with continuous scores; bars clipped at -40% and 140%')
a_ = fig.add_subplot(gs[0, 2]); rowsB = [('EyePACS, severe DR', B_['EyePACS_severe']['all']), ('DDR, PDR', B_['DDR_pdr']['all']), ('DDR, severe DR', B_['DDR_severe']['all']),
                                          ('BRSET, EyePACS-trained', B_['BRSET_severe']['EyePACS']), ('BRSET, DDR-trained', B_['BRSET_severe']['DDR'])]
for i, (lab, r) in enumerate(rowsB):
    a_.plot([r['worst_gain'], r['best_gain']], [i, i], color=BLUE, lw=2.2, solid_capstyle='round'); a_.plot(r['mean_gain_vs_vendor_alone'], i, 'o', color=BLUE, ms=7, mec='white', zorder=3)
    a_.text(r['best_gain'] + 2, i, f"{100 * r['share_vendors_helped']:.0f}% of {r['vendors']} helped", va='center', fontsize=7.2)
a_.axvline(0, color='#333333', lw=0.8); a_.set_yticks(range(len(rowsB))); a_.set_yticklabels([r[0] for r in rowsB], fontsize=7.8); a_.invert_yaxis()
a_.set_xlim(right=max(r['best_gain'] for _, r in rowsB) * 1.6); a_.set_xlabel('Missed per 10,000 avoided against the system alone')
panel(a_, 'b', 'A fixed background partner', f"DenseNet121 five-class (EyePACS), validation AUC {B_['partner']['validation_auc']:.3f}")
for j, (rule, lt) in enumerate((('max_rank_vs_best', 'c'), ('mean_rank_vs_best', 'd'))):
    a_ = fig.add_subplot(gs[1, j])
    for k, v in C_.items():
        ks = sorted(int(x) for x in v)
        a_.plot(ks, [v[str(x)][rule] for x in ks], marker=KM(k), ms=3.5, lw=1.5, color=KC[k], label=LBL_DEP[k])
    a_.axhline(0, color='#333333', lw=0.8); a_.set_xlabel('Systems combined'); a_.set_ylabel('Missed per 10,000 avoided vs the best member')
    panel(a_, lt, 'More than two systems: ' + ('maximum rank' if lt == 'c' else 'mean rank'), 'Up to 200 random groups of each size')
a_.legend(fontsize=6.8, loc='upper left', bbox_to_anchor=(1.03, 1.0))
title(fig, 'Supplementary Figure 12. Deployment checks: categorical outputs, a fixed background partner and more than two systems (exploratory, post hoc)')
save(fig, 'SuppFig12_deployment')

saved SuppFig12_deployment


In [16]:
# Supplementary Figure 13: pooled training versus pairing (notebook 26; plan dated before training)
ARMS = [('single_E', 'EyePACS-only single', '#9ecae1'), ('single_D', 'DDR-only single', '#fdae6b'), ('single_P', 'Pooled single', '#a1d99b'),
        ('pair_mixed_E_D', 'EyePACS-trained + DDR-trained pair', ORANGE), ('pair_pooled', 'Pooled pair', GREEN), ('pair_E', 'EyePACS-only pair', BLUE)]
SETS = [('BRSET | severe', 'BRSET (external to every model)'), ('EyePACS fresh | severe', 'EyePACS test set'), ('DDR test | severe', 'DDR official test file')]
fig, axes = plt.subplots(1, 3, figsize=(15, 4.6))
for a_, (key, ttl), lt in zip(axes, SETS, 'abc'):
    r = POOL[key]['missed_per_10k']
    for i, (k, lab, c) in enumerate(ARMS):
        m, lo, hi = r[k]; a_.bar(i, m, 0.75, color=c); a_.errorbar(i, m, yerr=[[max(0, m - lo)], [max(0, hi - m)]], color='#333333', lw=1, capsize=2.5)
    a_.set_xticks(range(len(ARMS))); a_.set_xticklabels([a[1] for a in ARMS], rotation=40, ha='right', fontsize=7.4); a_.set_ylabel('Missed severe DR per 10,000 images')
    panel(a_, lt, ttl)
v = POOL['verdicts_brset_severe']
axes[0].text(0, -0.62, 'Pre-specified (BRSET): ' + '; '.join(f'{k.split()[0]} {s}' for k, s in v.items()), transform=axes[0].transAxes, fontsize=7.6, color='#444444')
title(fig, 'Supplementary Figure 13. Pooled training versus pairing (same referral share; singles averaged over 12 models; pairs combine two different designs; 95% patient bootstrap)')
fig.tight_layout(); save(fig, 'SuppFig13_pooled_training')

saved SuppFig13_pooled_training


In [17]:
# Check: the recomputed numbers equal the ones behind the previous figures
OLD = json.load(open(f'{OUT}/final_figures_data_v2.json'))
ok = abs(OLD['miss_rate_law_spearman'] - round(float(rho), 3)) < 1e-9 and OLD['omitted_points'] == DROPPED and OLD['sibling_detector'] == DET and OLD['benefit_by_kappa_band'] == {k: BEN[k] for k in OLD['benefit_by_kappa_band']}
print('same numbers as final_figures_data_v2.json:', ok)
for k in F3: print(k, 'same minus different data:', ci_txt(F3[k]['mean'], *F3[k]['ci95']))
json.dump({'generated_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(), 'status': 'figures after review cycle 1; aggregates only',
           'fig2_rows': F2, 'fig3_brset_referable': BRH3, 'miss_rate_law_spearman': round(float(rho), 3), 'omitted_points': DROPPED, 'sibling_detector': DET,
           'benefit_by_kappa_band': BEN, 'fig7': F7, 'same_numbers_as_v2': bool(ok)}, open(f'{OUT}/figures_c1_data.json', 'w'), indent=1, default=float)
drive.flush_and_unmount(); drive.mount('/content/drive')
print('Wrote 05_figures/manuscript_final: Fig1 to Fig7, SuppFig9 to SuppFig13, figures_c1_data.json; previous versions renamed SUPERSEDED_')

same numbers as final_figures_data_v2.json: True
HIBA_mel same minus different data: 0.44 (0.23 to 0.75)
HIBA_mal same minus different data: 0.88 (0.60 to 1.20)
PAD_mal same minus different data: 0.45 (0.37 to 0.54)
BRSET same minus different data: 0.39 (0.32 to 0.46)
Mounted at /content/drive
Wrote 05_figures/manuscript_final: Fig1 to Fig7, SuppFig9 to SuppFig13, figures_c1_data.json; previous versions renamed SUPERSEDED_
